<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 1 &middot; Week 2 &middot; Part 1 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Python, NumPy and Vectorization

Week 1 got by with eight training examples and one feature. From today a training example
is a *vector* and the training set is a *matrix*, and the only way to work with those at any
speed is NumPy. This lab is the tool sharpening: no new machine learning, but every line of
the next three labs is written in what you build here.

**Companion theory lecture:** CSE 815, Week 2, Part 1 (Multiple Features and Vectorization).

## What you will be able to do

1. Create NumPy arrays, and state the `shape` of any expression before running it.
2. Index and slice 1-D and 2-D arrays, and say which one you have.
3. Use broadcasting deliberately, including per-column statistics with `axis=0`.
4. Implement the dot product with a loop and with `np.dot`, and measure the difference.
5. Recognise the `(m,)` versus `(m,1)` bug before it silently corrupts your results.

---

## 1. Arrays and their shape

A NumPy array is a grid of numbers of one type, plus a `shape` describing the grid. The
`shape` is the single most useful thing to print when something goes wrong.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

np.set_printoptions(precision=4, suppress=True)

a = np.zeros(4)                      # 4 zeros
b = np.ones(4)
c = np.array([2.0, 4.5, 1.0, 3.5])   # from a Python list
d = np.arange(4.0)                   # 0, 1, 2, 3

for name, arr in [("zeros", a), ("ones", b), ("array", c), ("arange", d)]:
    print(f"{name:8s} {str(arr):28s} shape={arr.shape}  ndim={arr.ndim}  dtype={arr.dtype}")

`shape=(4,)` is a **1-D array of length 4**. The trailing comma is Python's way of writing a
one-element tuple; it is not a typo and it is not a missing second dimension.

A 2-D array &mdash; a matrix &mdash; has `shape=(rows, columns)`. This is how a training set
with several features is stored: one **row per example**, one **column per feature**.

In [ ]:
# A tiny training set: 3 flats, 2 features (size in k sq ft, number of bedrooms).
X = np.array([[2.10, 5.0],
              [1.42, 3.0],
              [0.85, 2.0]])

print("X =")
print(X)
print()
print("shape :", X.shape, " -> m =", X.shape[0], "examples,  n =", X.shape[1], "features")
print("ndim  :", X.ndim)
print("size  :", X.size, "numbers in total")

> **The naming convention used for the rest of the course.** Capital `X` is the 2-D matrix of
> features, lower-case `y` the 1-D vector of targets, `w` the 1-D vector of weights, `b` a
> scalar. Sticking to it makes shape bugs visible by eye.

---

## 2. Indexing and slicing

Indexing picks out one element. Slicing picks out a range, and the endpoint is **excluded**.

In [ ]:
v = np.array([10., 20., 30., 40., 50., 60.])

print("v          :", v)
print("v[0]       :", v[0])            # first
print("v[-1]      :", v[-1])           # last
print("v[2:5]     :", v[2:5])          # elements 2, 3, 4 -- not 5
print("v[:3]      :", v[:3])           # from the start
print("v[3:]      :", v[3:])           # to the end
print("v[::2]     :", v[::2])          # every second element
print()
print("v[2:5].shape :", v[2:5].shape, " -- a slice of a 1-D array is 1-D")
print("v[2].shape   :", np.shape(v[2]), " -- a single element is a scalar, not an array")

In [ ]:
# 2-D indexing takes a row index and a column index.
print("X =")
print(X, "\n")

print("X[1, 0]  (example 1, feature 0) :", X[1, 0])
print("X[1]     (all of example 1)     :", X[1], " shape", X[1].shape)
print("X[:, 0]  (feature 0, every row) :", X[:, 0], " shape", X[:, 0].shape)
print("X[:2]    (first two examples)   :\n", X[:2])

Two habits worth forming now:

- `X[i]` is example $i$ &mdash; the notation the lecture writes as $\mathbf{x}^{(i)}$.
- `X[:, j]` is feature $j$ across every example. `X[i, j]` is $x^{(i)}_j$, one cell.

Note that both `X[1]` and `X[:, 0]` come back as **1-D** arrays. NumPy drops the dimension you
indexed with a single integer. That is convenient and it is also the source of the bug in
section 6.

---

## 3. Element-wise operations and broadcasting

Arithmetic on arrays is element-wise. No loop, no `map`.

In [ ]:
p = np.array([1., 2., 3., 4.])
q = np.array([10., 20., 30., 40.])

print("p + q :", p + q)
print("p * q :", p * q)          # ELEMENT-WISE, not a dot product
print("q / p :", q / p)
print("p ** 2:", p ** 2)
print("-p    :", -p)
print()
print("2 * p    :", 2 * p)       # scalar and array
print("p + 100  :", p + 100)

The scalar cases are the simplest instance of **broadcasting**: NumPy stretched the single
number `2` across all four elements. The general rule compares shapes from the right and
stretches any axis of length 1.

For this course the case that matters is a `(m, n)` matrix against an `(n,)` vector: the
vector is applied to **every row**.

In [ ]:
X_big = np.array([[2.10, 5.0, 45.0],
                  [1.42, 3.0, 40.0],
                  [0.85, 2.0, 36.0],
                  [1.94, 4.0, 15.0]])
print("X_big.shape :", X_big.shape)

# Per-feature statistics: axis=0 collapses the ROWS, leaving one number per COLUMN.
mu = X_big.mean(axis=0)
sigma = X_big.std(axis=0)
print("mu    :", mu, " shape", mu.shape)
print("sigma :", sigma, " shape", sigma.shape)

# (4, 3) - (3,)  ->  the (3,) vector is broadcast across all 4 rows.
X_centred = X_big - mu
print("\nX_big - mu =")
print(X_centred)
print("\ncolumn means of the result:", X_centred.mean(axis=0).round(12))

> **`axis=0` versus `axis=1`.** `axis=0` runs *down* the columns and gives one statistic per
> feature &mdash; almost always what you want. `axis=1` runs *across* the row and gives one
> number per example, which for a feature matrix is meaningless (it would average square feet
> with bedrooms). Getting this wrong is the single most common feature-scaling bug, and it
> raises no error.

In [ ]:
print("mean(axis=0) -- per feature :", X_big.mean(axis=0), " shape", X_big.mean(axis=0).shape)
print("mean(axis=1) -- per example :", X_big.mean(axis=1), " shape", X_big.mean(axis=1).shape)
print("mean()       -- everything  :", X_big.mean())

---

## 4. The dot product

The model of Week 2 is $f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w}\cdot\mathbf{x} + b$,
and the dot product is

$$\mathbf{w}\cdot\mathbf{x} = \sum_{j=1}^{n} w_j x_j .$$

It takes two vectors of the same length and returns **one number**. Write it three ways and
check they agree.

In [ ]:
def dot_loop(a, b):
    """Dot product of two 1-D arrays, computed with an explicit loop."""
    assert a.shape == b.shape, f"shapes differ: {a.shape} vs {b.shape}"
    total = 0.0
    for j in range(a.shape[0]):
        total += a[j] * b[j]
    return total


def dot_elementwise(a, b):
    """Dot product via an element-wise product followed by a sum."""
    return np.sum(a * b)


w = np.array([0.1, 4.0, 10.0, -2.0])
x = np.array([1.5, 3.0, 2.0, 20.0])

print("loop        :", dot_loop(w, x))
print("sum(a*b)    :", dot_elementwise(w, x))
print("np.dot      :", np.dot(w, x))
print("w @ x       :", w @ x)          # @ is the same operator, and reads better

assert np.allclose(dot_loop(w, x), np.dot(w, x))
assert np.allclose(dot_elementwise(w, x), np.dot(w, x))
print("\nAll three agree.")

That is the lecture's worked example: with $\mathbf{w} = [0.1, 4, 10, -2]$, $b = 80$ and a
flat $\mathbf{x} = [1.5, 3, 2, 20]$, the model predicts

In [ ]:
b = 80.0
f = np.dot(w, x) + b
print(f"f(x) = w . x + b = {np.dot(w, x):.2f} + {b:.0f} = {f:.2f} lakh BDT")

# Read the four contributions separately -- this is what makes a linear model interpretable.
labels = ["size (k sq ft)", "bedrooms", "floors", "age (yr)"]
print()
for j in range(4):
    print(f"  {labels[j]:16s} w={w[j]:6.1f} * x={x[j]:5.1f}  contributes {w[j]*x[j]:8.2f} lakh")
print(f"  {'base (b)':16s} {'':19s} contributes {b:8.2f} lakh")

### Checkpoint 1

Write `dot_check(a, b)` that returns `True` when your loop implementation and `np.dot` agree
to within `np.allclose`'s default tolerance, and `False` otherwise. Test it on
`a = np.arange(1.0, 6.0)` and `b = np.array([2., 0., -1., 4., 0.5])`.

*Expected:* the dot product is `17.5`, and `dot_check` returns `True`.

In [ ]:
# Your code here

---

## 5. Why vectorization is worth the trouble

The vectorised version is shorter. That is not the reason to use it. `np.dot` hands the arrays
to compiled code that uses the CPU's SIMD instructions &mdash; one instruction operating on
several numbers at once &mdash; and then reduces the products in parallel. The Python loop
executes one multiply per interpreter step.

Measure it. The numbers below depend on your machine; the **ratio** is the point.

In [ ]:
import time

rng = np.random.default_rng(0)
N = 1_000_000
big_a = rng.random(N)
big_b = rng.random(N)

t0 = time.time()
r_loop = dot_loop(big_a, big_b)
t_loop = time.time() - t0

t0 = time.time()
r_np = np.dot(big_a, big_b)
t_np = time.time() - t0

print(f"N = {N:,}")
print(f"  loop    : {r_loop:.6f}   in {t_loop*1000:9.3f} ms")
print(f"  np.dot  : {r_np:.6f}   in {t_np*1000:9.3f} ms")
print(f"\n  speed-up: {t_loop/t_np:,.0f} times")

assert np.allclose(r_loop, r_np), "the two dot products disagree"

Two things to take from that number.

- The results agree to `np.allclose` but not bit-for-bit. `np.dot` sums in a different order,
  and floating-point addition is not associative. This is normal and is exactly why we never
  compare floats with `==`.
- The speed-up grows with the size of the data. Gradient descent evaluates a dot product for
  every example on every iteration, so a 10,000-iteration run multiplies this gap by 10,000.

### The same argument applies to the parameter update

The gradient descent step $w_j := w_j - \alpha\, d_j$ for all $j$ is one line, not a loop.

In [ ]:
w_demo = np.array([1.0, 2.0, 3.0, 4.0])
d_demo = np.array([0.5, -1.0, 2.0, 0.0])
alpha = 0.1

# Without vectorization
w_a = w_demo.copy()
for j in range(w_a.shape[0]):
    w_a[j] = w_a[j] - alpha * d_demo[j]

# With vectorization
w_b = w_demo - alpha * d_demo

print("loop       :", w_a)
print("vectorised :", w_b)
assert np.allclose(w_a, w_b)
print("\nSame answer, one line.")

---

## 6. Shapes are a contract

Most NumPy bugs in this course are shape bugs, and the dangerous ones raise no error. Two to
learn to recognise now.

### `*` is not `np.dot`

In [ ]:
u = np.array([1., 2., 3.])
v = np.array([4., 5., 6.])

print("u * v      :", u * v,      " shape", (u * v).shape,      "  <- a VECTOR")
print("np.dot(u,v):", np.dot(u, v), " shape", np.shape(np.dot(u, v)), "         <- a SCALAR")

Writing `f = w * x + b` instead of `f = np.dot(w, x) + b` gives an array where a number was
expected. Nothing raises. The error surfaces much later, as a cost function that returns an
array, or a broadcast that silently produces an $m\times m$ matrix.

### `(m,)` is not `(m, 1)`

In [ ]:
y_flat = np.array([30., 36., 45.])          # shape (3,)   -- what we use throughout
y_col  = y_flat.reshape(-1, 1)              # shape (3, 1) -- a column matrix

print("y_flat shape:", y_flat.shape)
print("y_col  shape:", y_col.shape)
print()

pred = np.array([31., 35., 47.])            # shape (3,)

print("pred - y_flat  ->", (pred - y_flat).shape, " correct: one residual per example")
print("pred - y_col   ->", (pred - y_col).shape,  " WRONG: broadcasting made a 3x3 matrix")
print()
print("pred - y_col =")
print(pred - y_col)

Nothing is flagged. `np.sum(...)` on the wrong one still returns a number, so a cost function
built on it returns a plausible-looking value that is simply not the cost.

> **The habit that prevents all of this:** print `.shape` while developing, and assert it in
> any function you will reuse. The next lab does exactly that.

In [ ]:
def residuals(pred, y):
    """Residuals pred - y, with the shape contract enforced."""
    assert pred.ndim == 1, f"pred must be 1-D, got shape {pred.shape}"
    assert y.ndim == 1,    f"y must be 1-D, got shape {y.shape}"
    assert pred.shape == y.shape, f"shape mismatch: {pred.shape} vs {y.shape}"
    return pred - y


print("good call :", residuals(pred, y_flat))

try:
    residuals(pred, y_col)
except AssertionError as e:
    print("bad call caught:", e)

### Checkpoint 2

Write `zscore(X)` that returns `(X_scaled, mu, sigma)` where each column of `X_scaled` has
mean 0 and standard deviation 1. Use `axis=0`. Apply it to `X_big` from section 3 and assert
that the column means are within `1e-12` of zero and the column standard deviations are
within `1e-12` of one.

*Expected:* `mu` is `[1.5775, 3.5, 34.0]` and `sigma` is approximately
`[0.4895, 1.1180, 11.4237]`.

In [ ]:
# Your code here

---

## 7. Recap

| Idea | Code |
|---|---|
| shape of an array | `a.shape` &mdash; print it whenever confused |
| one example, one feature | `X[i]`, `X[:, j]`, `X[i, j]` |
| per-feature statistic | `X.mean(axis=0)` |
| element-wise product | `a * b` &rarr; a vector |
| dot product | `np.dot(a, b)` or `a @ b` &rarr; a scalar |
| parameter update | `w = w - alpha * d` |

- A slice keeps the dimension; a single integer index drops it.
- Broadcasting applies an `(n,)` vector to every row of an `(m, n)` matrix.
- `(m,)` minus `(m, 1)` gives an $m \times m$ matrix and no warning.
- Vectorised code is faster because of SIMD hardware, not because loops are "slow Python".

### Exercises to hand in

1. Time `dot_loop` against `np.dot` for `N` in `[10**3, 10**4, 10**5, 10**6]`. Plot both
   times against `N` on log&ndash;log axes. Both should be straight lines of slope 1; explain
   what the vertical gap between them represents and why it does not close as `N` grows.
2. Implement `matvec_loop(X, w)` returning $X\mathbf{w}$ with two nested loops, and confirm
   it matches `X @ w` for a random `X` of shape `(200, 8)`. Report the speed-up.
3. Construct a case where `X.mean(axis=1)` runs without error but produces a meaningless
   number, using the `X_big` matrix. State in one sentence what the number you computed
   actually is.
4. `a = np.array([1., 2., 3.])` and `B = np.array([[1.], [2.], [3.]])`. Predict the shape of
   `a + B`, `a * B` and `np.dot(a, B)` **before** running them. Run them and explain any
   prediction you got wrong.
5. Floating-point order matters: build `x = np.full(10**6, 0.1)` and compare
   `dot_loop(x, np.ones(10**6))` with `np.dot(x, np.ones(10**6))` and with `10**5`. Report
   all three to 12 decimal places and explain the differences.

### Next

**Part 2 &mdash; Multiple Linear Regression:** the real data set, four features, and a
vectorised cost and gradient built entirely out of what you wrote today.

**Reading:** James et al., *ISL* 2e, &sect;3.2.